In [8]:
# Diagnostic — run this BEFORE anything else in gold notebook
check = spark.read.table("silver_lakehouse.dbo.silver_delivery_analysis")
print(f"Row count: {check.count():,}")
print(f"Columns ({len(check.columns)}): {check.columns}")

# Specifically check for the Cell 14 columns
cell14_cols = ["is_interstate", "purchase_year", "purchase_month", 
               "purchase_quarter", "is_weekend_purchase"]
for c in cell14_cols:
    status = "✅" if c in check.columns else "❌  MISSING"
    print(f"  {status}  {c}")

StatementMeta(, 5fc7dcbb-6f32-488a-a30f-2122a8550e52, 10, Finished, Available, Finished, False)

Row count: 103,200
Columns (48): ['seller_id', 'customer_id', 'product_id', 'order_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'order_item_id', 'shipping_limit_date', 'price', 'freight_value', 'product_category_name', 'product_name_length', 'product_description_length', 'product_photos_qty', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state', 'seller_zip_code_prefix', 'seller_city', 'seller_state', 'delay_days', 'delivery_status', 'delay_bucket', 'volume_cm3', 'weight_kg', 'freight_band', 'route', 'customer_lat', 'customer_lng', 'seller_lat', 'seller_lng', 'distance_km', 'distance_band', 'is_interstate', 'purchase_year', 'purchase_month', 'purchase_quarter', 'purchase_day_of_week', 'is_weekend_purchase']
  ✅  is_interstate
  ✅  purchase_year
  ✅  purc

In [9]:
### !!! RUN THIS CELL after diagnostic cell above !!! ###

from pyspark.sql import functions as F
from pyspark.sql.window import Window
 
analysis_df = spark.read.table("silver_lakehouse.dbo.silver_delivery_analysis")
print(f"Source: silver_delivery_analysis — {analysis_df.count():,} rows")
 
def write_gold(df, table_name: str):
    full_name = f"gold_lakehouse.dbo.gold_{table_name}"
    (df.write
       .format("delta")
       .mode("overwrite")
       .option("overwriteSchema", "true")
       .saveAsTable(full_name))
    count = spark.read.table(full_name).count()
    print(f"✅  {full_name}: {count:,} rows written")

StatementMeta(, 5fc7dcbb-6f32-488a-a30f-2122a8550e52, 11, Finished, Available, Finished, False)

Source: silver_delivery_analysis — 103,200 rows


In [10]:
## 1.0 Create Dimdate(dimension) table

from pyspark.sql.types import StructType, StructField, DateType
import datetime
 
# 1.1 Create a complete date spine from the earliest to latest order date
min_date, max_date = (
    analysis_df
    .agg(
        F.min("order_purchase_timestamp").alias("min_d"),
        F.max("order_purchase_timestamp").alias("max_d")
    )
    .collect()[0]
)

# 1.2 Create upper & lower limit for dates
min_date = min_date.date() if min_date else datetime.date(2016, 1, 1)
max_date = max_date.date() if max_date else datetime.date(2018, 12, 31)
 
date_list = [
    (min_date + datetime.timedelta(days=i),)
    for i in range((max_date - min_date).days + 1)
]

# 1.3 Creating dataframe to contain the variables created
dim_date_df = spark.createDataFrame(date_list, ["full_date"])
dim_date_df = (
    dim_date_df
    .withColumn("year",         F.year("full_date"))
    .withColumn("month",        F.month("full_date"))
    .withColumn("quarter",      F.quarter("full_date"))
    .withColumn("day_of_week",  F.dayofweek("full_date"))     # 1=Sun, 7=Sat
    .withColumn("day_name",     F.date_format("full_date", "EEEE"))
    .withColumn("month_name",   F.date_format("full_date", "MMMM"))
    .withColumn("is_weekend",   F.when(F.dayofweek("full_date").isin(1, 7), True).otherwise(False))
    .withColumn("year_month",   F.date_format("full_date", "yyyy-MM"))
)


 # 1.4 Validte if dim_date_df is correctly created
write_gold(dim_date_df, "dim_date")
dim_date_df.show(5, truncate=False)

StatementMeta(, 5fc7dcbb-6f32-488a-a30f-2122a8550e52, 12, Finished, Available, Finished, False)

✅  gold_lakehouse.dbo.gold_dim_date: 774 rows written
+----------+----+-----+-------+-----------+---------+----------+----------+----------+
|full_date |year|month|quarter|day_of_week|day_name |month_name|is_weekend|year_month|
+----------+----+-----+-------+-----------+---------+----------+----------+----------+
|2016-09-04|2016|9    |3      |1          |Sunday   |September |true      |2016-09   |
|2016-09-05|2016|9    |3      |2          |Monday   |September |false     |2016-09   |
|2016-09-06|2016|9    |3      |3          |Tuesday  |September |false     |2016-09   |
|2016-09-07|2016|9    |3      |4          |Wednesday|September |false     |2016-09   |
|2016-09-08|2016|9    |3      |5          |Thursday |September |false     |2016-09   |
+----------+----+-----+-------+-----------+---------+----------+----------+----------+
only showing top 5 rows



In [11]:
## 2.0 Create DimCustomer table

# 2.1 Extraction of columns from silver table
dim_customer_df = (
    spark.read.table("silver_lakehouse.dbo.silver_customers")
    .select(
        F.col("customer_id"),
        F.col("customer_unique_id"),
        F.col("customer_zip_code_prefix"),
        F.col("customer_city"),
        F.col("customer_state"),
    )
    .dropDuplicates(["customer_id"])
)
 
# 2.2 Validte if columns are correctly created
write_gold(dim_customer_df, "dim_customer")
dim_customer_df.show(5, truncate=False)

StatementMeta(, 5fc7dcbb-6f32-488a-a30f-2122a8550e52, 13, Finished, Available, Finished, False)

✅  gold_lakehouse.dbo.gold_dim_customer: 99,441 rows written
+--------------------------------+--------------------------------+------------------------+-------------------+--------------+
|customer_id                     |customer_unique_id              |customer_zip_code_prefix|customer_city      |customer_state|
+--------------------------------+--------------------------------+------------------------+-------------------+--------------+
|00050bf6e01e69d5c0fd612f1bcfb69c|e3cf594a99e810f58af53ed4820f25e5|98700                   |ijui               |RS            |
|000598caf2ef4117407665ac33275130|7e0516b486e92ed3f3afdd6d1276cfbd|35540                   |oliveira           |MG            |
|0013cd8e350a7cc76873441e431dd5ee|334fed5abcee3aa96c13f1432703e1fd|3585                    |sao paulo          |SP            |
|0015bc9fd2d5395446143e8b215d7c75|490c854539b21598cfbbac518ca25788|12233                   |sao jose dos campos|SP            |
|001df1ee5c36767aa607001ab1a13a06|46b44ab32

In [12]:
## 3.0 Create DimCustomer table

# 3.1 Extraction of columns from silver table
dim_seller_df = (
    spark.read.table("silver_lakehouse.dbo.silver_sellers")
    .select(
        F.col("seller_id"),
        F.col("seller_zip_code_prefix"),
        F.col("seller_city"),
        F.col("seller_state"),
    )
    .dropDuplicates(["seller_id"])
)

# 3.2 Validte if columns are correctly created
write_gold(dim_seller_df, "dim_seller")
dim_seller_df.show(5, truncate=False)

StatementMeta(, 5fc7dcbb-6f32-488a-a30f-2122a8550e52, 14, Finished, Available, Finished, False)

✅  gold_lakehouse.dbo.gold_dim_seller: 3,095 rows written
+--------------------------------+----------------------+-----------+------------+
|seller_id                       |seller_zip_code_prefix|seller_city|seller_state|
+--------------------------------+----------------------+-----------+------------+
|0015a82c2db000af6aaaf3ae2ecb0532|9080                  |santo andre|SP          |
|001cca7ae9ae17fb1caed9dfb1094831|29156                 |cariacica  |ES          |
|001e6ad469a905060d959994f1b41e4f|24754                 |sao goncalo|RJ          |
|002100f778ceb8431b7a1020ff7ab48f|14405                 |franca     |SP          |
|003554e2dce176b5555353e4f3555ac8|74565                 |goiania    |GO          |
+--------------------------------+----------------------+-----------+------------+
only showing top 5 rows



In [13]:
## 4.0 Create DimProduct table 

# 4.1 Extract columns and via joining product & product translation table
dim_product_df = (
    spark.read.table("silver_lakehouse.dbo.silver_products")
    .join(
        spark.read.table("silver_lakehouse.dbo.silver_product_category_name_translation")
            .select("product_category_name", "product_category_name_english"),
        on="product_category_name",
        how="left"
    )
    .select(
        F.col("product_id"),
        F.col("product_category_name"),
        F.col("product_category_name_english"),
        F.col("product_weight_g"),
        F.col("product_length_cm"),
        F.col("product_height_cm"),
        F.col("product_width_cm"),
        (F.col("product_length_cm") * F.col("product_height_cm") * F.col("product_width_cm"))
            .alias("product_volume_cm3"),
        F.round(F.col("product_weight_g") / 1000, 3).alias("product_weight_kg"),
    )
    .dropDuplicates(["product_id"])
)

# 4.2 Validate if columns are create correctly
write_gold(dim_product_df, "dim_product")
dim_product_df.show(5, truncate=False)

StatementMeta(, 5fc7dcbb-6f32-488a-a30f-2122a8550e52, 15, Finished, Available, Finished, False)

✅  gold_lakehouse.dbo.gold_dim_product: 32,951 rows written
+--------------------------------+---------------------+-----------------------------+----------------+-----------------+-----------------+----------------+------------------+-----------------+
|product_id                      |product_category_name|product_category_name_english|product_weight_g|product_length_cm|product_height_cm|product_width_cm|product_volume_cm3|product_weight_kg|
+--------------------------------+---------------------+-----------------------------+----------------+-----------------+-----------------+----------------+------------------+-----------------+
|00066f42aeeb9f3007548bb9d3f33c38|perfumaria           |perfumery                    |300             |20               |16               |16              |5120              |0.3              |
|00088930e925c41fd95ebfe695fd2655|automotivo           |auto                         |1225            |55               |10               |26              |14300   

In [14]:
## 5.0 Create FactDelivery table

# 5.1 Extract columns from analysis_df and insert into fact_df
fact_df = (
    analysis_df
    .select(
        # Keys
        F.col("order_id"),
        F.col("order_item_id"),
        F.col("customer_id"),
        F.col("seller_id"),
        F.col("product_id"),
 
        # Date FK — join to DimDate on this
        F.to_date(F.col("order_purchase_timestamp")).alias("purchase_date"),
        F.to_date(F.col("order_delivered_customer_date")).alias("delivered_date"),
        F.to_date(F.col("order_estimated_delivery_date")).alias("estimated_delivery_date"),
 
        # Order status
        F.col("order_status"),
 
        # Delivery metrics
        F.col("delay_days"),
        F.col("delivery_status"),
        F.col("delay_bucket"),
        F.col("is_interstate"),
 
        # Freight
        F.col("freight_value"),
        F.col("freight_band"),
        F.col("price"),
 
        # Product metrics
        F.col("product_weight_g"),
        F.col("weight_kg"),
        F.col("volume_cm3"),
 
        # Geography
        F.col("seller_state"),
        F.col("customer_state"),
        F.col("route"),
        F.col("distance_km"),
        F.col("distance_band"),
        F.col("seller_lat"),
        F.col("seller_lng"),
        F.col("customer_lat"),
        F.col("customer_lng"),
 
        # Time dimensions (pre-computed for BI performance)
        F.col("purchase_year"),
        F.col("purchase_month"),
        F.col("purchase_quarter"),
        F.col("is_weekend_purchase"),
    )
)
 
# 5.2 Create column for late flag as integer to be use for easy SUM aggregation in Power BI
fact_df = fact_df.withColumn(
    "is_late",
    F.when(F.col("delivery_status") == "Late", 1).otherwise(0)
)
 
# 5.3 Create column for total order value per line item
fact_df = fact_df.withColumn(
    "total_line_value",
    F.col("price") + F.col("freight_value")
)
 
write_gold(fact_df, "fact_delivery")

StatementMeta(, 5fc7dcbb-6f32-488a-a30f-2122a8550e52, 16, Finished, Available, Finished, False)

✅  gold_lakehouse.dbo.gold_fact_delivery: 103,200 rows written


In [15]:
# Validation Checking for gold table

gold_fact = spark.read.table("gold_fact_delivery")
 
print(f"\n--- Gold FactDelivery summary ---")
print(f"Total rows:         {gold_fact.count():,}")
 
print("\nDelivery status breakdown:")
gold_fact.groupBy("delivery_status").count().orderBy("delivery_status").show()
 
print("Late rate by distance band:")
(gold_fact
 .filter(F.col("delivered_date").isNotNull())
 .groupBy("distance_band")
 .agg(
     F.count("*").alias("total_orders"),
     F.sum("is_late").alias("late_orders"),
     F.round(F.avg("delay_days"), 2).alias("avg_delay_days"),
     F.round(F.avg("freight_value"), 2).alias("avg_freight"),
     F.round(F.avg("distance_km"), 1).alias("avg_distance_km")
 )
 .withColumn("late_rate_pct", F.round(F.col("late_orders") / F.col("total_orders") * 100, 2))
 .orderBy("distance_band")
 .show()
)
 
print("Late rate by weight category:")
(gold_fact
 .filter(F.col("delivered_date").isNotNull())
 .withColumn("weight_band",
     F.when(F.col("weight_kg") < 0.5,  "< 0.5 kg")
      .when(F.col("weight_kg") < 2.0,  "0.5–2 kg")
      .when(F.col("weight_kg") < 5.0,  "2–5 kg")
      .otherwise("5+ kg"))
 .groupBy("weight_band")
 .agg(
     F.count("*").alias("total_orders"),
     F.sum("is_late").alias("late_orders"),
 )
 .withColumn("late_rate_pct", F.round(F.col("late_orders") / F.col("total_orders") * 100, 2))
 .orderBy("weight_band")
 .show()
)
 
print("\n🎉  Gold layer complete.")
print("    Tables ready for Power BI:")
print("    - gold_fact_delivery")
print("    - gold_dim_date")
print("    - gold_dim_customer")
print("    - gold_dim_seller")
print("    - gold_dim_product")

StatementMeta(, 5fc7dcbb-6f32-488a-a30f-2122a8550e52, 17, Finished, Available, Finished, False)


--- Gold FactDelivery summary ---
Total rows:         103,200

Delivery status breakdown:
+-----------------+-----+
|  delivery_status|count|
+-----------------+-----+
|            Early|92199|
|             Late| 6666|
|Not Delivered Yet| 3005|
|          On Time| 1330|
+-----------------+-----+

Late rate by distance band:
+-------------+------------+-----------+--------------+-----------+---------------+-------------+
|distance_band|total_orders|late_orders|avg_delay_days|avg_freight|avg_distance_km|late_rate_pct|
+-------------+------------+-----------+--------------+-----------+---------------+-------------+
|      0-50 km|       12159|        527|         -9.55|      11.46|           21.5|         4.33|
|   100-500 km|       38517|       2298|        -12.21|      18.39|          322.6|         5.97|
|     1000+ km|       16346|       1686|        -12.81|       31.5|         1736.1|        10.31|
|    50-100 km|        6378|        281|         -9.71|      12.38|           75.4| 

In [1]:
# Fix distance_band sort order — add numeric sort column to fact table
from pyspark.sql import functions as F

gold_fact = spark.read.table("gold_lakehouse.dbo.gold_fact_delivery")

gold_fact = gold_fact.withColumn(
    "distance_sort_order",
    F.when(F.col("distance_band") == "0-50 km",     1)
     .when(F.col("distance_band") == "50-100 km",   2)
     .when(F.col("distance_band") == "100-500 km",  3)
     .when(F.col("distance_band") == "500-1000 km", 4)
     .when(F.col("distance_band") == "1000+ km",    5)
     .otherwise(6)
)

# Also fix delay_bucket sort order while we're here
gold_fact = gold_fact.withColumn(
    "delay_bucket_sort_order",
    F.when(F.col("delay_bucket") == "Not Late",  1)
     .when(F.col("delay_bucket") == "1-3 Days",  2)
     .when(F.col("delay_bucket") == "4-7 Days",  3)
     .when(F.col("delay_bucket") == "8-14 Days", 4)
     .when(F.col("delay_bucket") == "15+ Days",  5)
     .otherwise(6)
)

(gold_fact.write
 .format("delta")
 .mode("overwrite")
 .option("overwriteSchema", "true")
 .saveAsTable("gold_lakehouse.dbo.gold_fact_delivery"))

print(f"✅  Updated gold_fact_delivery with sort columns")
print(f"    Row count: {spark.read.table('gold_lakehouse.dbo.gold_fact_delivery').count():,}")

StatementMeta(, 8bac11b1-2f09-4837-9810-0357c3595d12, 3, Finished, Available, Finished, False)

✅  Updated gold_fact_delivery with sort columns
    Row count: 103,200


In [1]:
# =============================================================
#  Add weight_band and sort columns to gold_fact_delivery
# =============================================================

from pyspark.sql import functions as F

gold_fact = spark.read.table("gold_lakehouse.dbo.gold_fact_delivery")

gold_fact = (gold_fact
    .withColumn(
        "weight_band",
        F.when(F.col("weight_kg") < 0.5,  "< 0.5 kg")
         .when(F.col("weight_kg") < 2.0,  "0.5-2 kg")
         .when(F.col("weight_kg") < 5.0,  "2-5 kg")
         .otherwise("5+ kg")
    )
    .withColumn(
        "weight_band_sort_order",
        F.when(F.col("weight_kg") < 0.5,  1)
         .when(F.col("weight_kg") < 2.0,  2)
         .when(F.col("weight_kg") < 5.0,  3)
         .otherwise(4)
    )
)

(gold_fact.write
 .format("delta")
 .mode("overwrite")
 .option("overwriteSchema", "true")
 .saveAsTable("gold_lakehouse.dbo.gold_fact_delivery"))

count = spark.read.table("gold_lakehouse.dbo.gold_fact_delivery").count()
print(f"✅  weight_band and weight_band_sort_order added")
print(f"    Row count: {count:,}")

# Verify new columns exist
spark.read.table("gold_lakehouse.dbo.gold_fact_delivery") \
    .select("weight_kg", "weight_band", "weight_band_sort_order") \
    .show(10)

StatementMeta(, 5f97b74d-09f1-4b0a-993c-0d98b84c45df, 3, Finished, Available, Finished, False)

✅  weight_band and weight_band_sort_order added
    Row count: 103,200
+---------+-----------+----------------------+
|weight_kg|weight_band|weight_band_sort_order|
+---------+-----------+----------------------+
|     0.45|   < 0.5 kg|                     1|
|      0.8|   0.5-2 kg|                     2|
|      9.8|      5+ kg|                     4|
|     0.75|   0.5-2 kg|                     2|
|    3.407|     2-5 kg|                     3|
|     0.45|   < 0.5 kg|                     1|
|      0.3|   < 0.5 kg|                     1|
|     8.25|      5+ kg|                     4|
|      0.3|   < 0.5 kg|                     1|
|      0.3|   < 0.5 kg|                     1|
+---------+-----------+----------------------+
only showing top 10 rows



In [2]:
# Verify weight_band columns exist in gold_fact_delivery
df = spark.read.table("gold_lakehouse.dbo.gold_fact_delivery")
print(f"Total columns: {len(df.columns)}")
print(f"All columns: {df.columns}")

# Check specifically for weight columns
weight_cols = [c for c in df.columns if "weight" in c.lower()]
print(f"\nWeight-related columns: {weight_cols}")

StatementMeta(, 5f97b74d-09f1-4b0a-993c-0d98b84c45df, 4, Finished, Available, Finished, False)

Total columns: 38
All columns: ['order_id', 'order_item_id', 'customer_id', 'seller_id', 'product_id', 'purchase_date', 'delivered_date', 'estimated_delivery_date', 'order_status', 'delay_days', 'delivery_status', 'delay_bucket', 'is_interstate', 'freight_value', 'freight_band', 'price', 'product_weight_g', 'weight_kg', 'volume_cm3', 'seller_state', 'customer_state', 'route', 'distance_km', 'distance_band', 'seller_lat', 'seller_lng', 'customer_lat', 'customer_lng', 'purchase_year', 'purchase_month', 'purchase_quarter', 'is_weekend_purchase', 'is_late', 'total_line_value', 'distance_sort_order', 'delay_bucket_sort_order', 'weight_band', 'weight_band_sort_order']

Weight-related columns: ['product_weight_g', 'weight_kg', 'weight_band', 'weight_band_sort_order']


In [1]:
# Full column audit of gold_fact_delivery
df = spark.read.table("gold_lakehouse.dbo.gold_fact_delivery")

print(f"Total rows: {df.count():,}")
print(f"Total columns: {len(df.columns)}")
print(f"\nAll columns:")
for col in sorted(df.columns):
    print(f"  - {col}")

StatementMeta(, 4be6cfbf-75e4-4a9e-877f-cb605ecf12e0, 3, Finished, Available, Finished, False)

Total rows: 103,200
Total columns: 38

All columns:
  - customer_id
  - customer_lat
  - customer_lng
  - customer_state
  - delay_bucket
  - delay_bucket_sort_order
  - delay_days
  - delivered_date
  - delivery_status
  - distance_band
  - distance_km
  - distance_sort_order
  - estimated_delivery_date
  - freight_band
  - freight_value
  - is_interstate
  - is_late
  - is_weekend_purchase
  - order_id
  - order_item_id
  - order_status
  - price
  - product_id
  - product_weight_g
  - purchase_date
  - purchase_month
  - purchase_quarter
  - purchase_year
  - route
  - seller_id
  - seller_lat
  - seller_lng
  - seller_state
  - total_line_value
  - volume_cm3
  - weight_band
  - weight_band_sort_order
  - weight_kg


In [2]:
spark.read.table("gold_lakehouse.dbo.gold_fact_delivery") \
    .select("delivery_status") \
    .distinct() \
    .show()

StatementMeta(, 4be6cfbf-75e4-4a9e-877f-cb605ecf12e0, 4, Finished, Available, Finished, False)

+-----------------+
|  delivery_status|
+-----------------+
|          On Time|
|Not Delivered Yet|
|             Late|
|            Early|
+-----------------+

